**Context.** This title cell identifies the notebook as the Problem 1 solution. The rest of the notebook is organized as setup, reusable model code, experiment execution, and interpretation.


# ECGR 4106 Homework 2 - Problem 1

**Context.** This preflight cell checks whether PyTorch imports successfully in Colab. It exists because some Colab runtimes can enter a broken PyTorch state; when that happens, the cell reinstalls the scientific stack and restarts the runtime before training begins.


In [1]:
# PyTorch import preflight for Colab
# If Colab's preinstalled PyTorch is in a broken state, this cell repairs it and restarts the runtime.
import importlib
import os
import subprocess
import sys

def ensure_torch_imports():
    try:
        return importlib.import_module("torch")
    except RuntimeError as exc:
        message = str(exc)
        known_colab_torch_import_bug = "THPDtypeType.tp_dict" in message or "Dtype.cpp" in message
        if not known_colab_torch_import_bug:
            raise
        print("PyTorch failed during import because the current runtime has a broken torch install.")
        print("Reinstalling PyTorch now. The runtime will restart automatically after installation.")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--upgrade",
                "--force-reinstall",
                "torch",
                "numpy",
                "pandas",
                "matplotlib",
            ]
        )
        os.kill(os.getpid(), 9)

torch_check = ensure_torch_imports()
print(f"PyTorch import check passed: torch {torch_check.__version__}")


PyTorch import check passed: torch 2.11.0+cu128


**Context.** This setup cell imports the required packages, fixes the random seed for repeatability, selects CUDA when a GPU is available, and creates an output folder for result tables.


In [2]:
# Colab-friendly setup
import math
import os
import random
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

OUTPUT_DIR = Path("/content/ECGR4106_HW2_outputs") if Path("/content").exists() else Path("ECGR4106_HW2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Using device: cuda


**Context.** The shared-code section defines the dataset, vocabulary, recurrent model, metric helpers, complexity estimates, training loop, and text generation helper used by the experiments.


## Shared Model, Dataset, Metrics, and Training Code

**Context.** This code cell implements the reusable character-level language modeling pipeline. It turns text into overlapping input-target windows, trains RNN/LSTM/GRU variants with the same evaluation procedure, and records the requested loss, accuracy, time, parameter count, model size, and approximate multiply-add complexity.


In [3]:
class CharWindowDataset(Dataset):
    def __init__(self, encoded_text, seq_len):
        self.data = torch.tensor(encoded_text, dtype=torch.long)
        self.seq_len = int(seq_len)
        if len(self.data) <= self.seq_len:
            raise ValueError("Text is too short for the requested sequence length.")

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


class CharVocabulary:
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    def encode(self, text):
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)

    def __len__(self):
        return len(self.chars)


def make_loaders(text, seq_len, batch_size=64, val_fraction=0.2):
    vocab = CharVocabulary(text)
    encoded = vocab.encode(text)
    split = max(seq_len + 1, int(len(encoded) * (1 - val_fraction)))
    train_encoded = encoded[:split]
    val_encoded = encoded[split - seq_len :]
    train_ds = CharWindowDataset(train_encoded, seq_len)
    val_ds = CharWindowDataset(val_encoded, seq_len)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, vocab


class CharLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        rnn_type="LSTM",
        embed_dim=64,
        hidden_size=128,
        num_layers=1,
        fc_hidden=0,
        dropout=0.0,
    ):
        super().__init__()
        self.rnn_type = rnn_type.upper()
        self.embed_dim = int(embed_dim)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.fc_hidden = int(fc_hidden)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[self.rnn_type]
        recurrent_dropout = dropout if num_layers > 1 else 0.0
        self.recurrent = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )
        if fc_hidden and fc_hidden > 0:
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_size, fc_hidden),
                nn.ReLU(),
                nn.Linear(fc_hidden, vocab_size),
            )
        else:
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, vocab_size))

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.recurrent(x, hidden)
        logits = self.head(out)
        return logits, hidden


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    total_bytes += sum(b.numel() * b.element_size() for b in model.buffers())
    return total_bytes / (1024 ** 2)


def recurrent_gate_count(rnn_type):
    return {"RNN": 1, "GRU": 3, "LSTM": 4}[rnn_type.upper()]


def approximate_recurrent_madds_per_sequence(rnn_type, seq_len, embed_dim, hidden_size, num_layers, vocab_size, fc_hidden=0):
    # Approximate multiply-add terms for one sample sequence. Biases and activations are omitted.
    gates = recurrent_gate_count(rnn_type)
    total = 0
    for layer in range(num_layers):
        input_dim = embed_dim if layer == 0 else hidden_size
        total += seq_len * gates * (input_dim * hidden_size + hidden_size * hidden_size)
    if fc_hidden and fc_hidden > 0:
        total += seq_len * (hidden_size * fc_hidden + fc_hidden * vocab_size)
    else:
        total += seq_len * hidden_size * vocab_size
    return int(total)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item() * y.numel()
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.numel()
    avg_loss = total_loss / total
    acc = correct / total
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, acc, ppl


def train_one_experiment(
    text,
    seq_len,
    rnn_type,
    *,
    epochs=10,
    batch_size=64,
    embed_dim=64,
    hidden_size=128,
    num_layers=1,
    fc_hidden=0,
    dropout=0.0,
    lr=0.003,
    val_fraction=0.2,
    label=None,
):
    train_loader, val_loader, vocab = make_loaders(text, seq_len, batch_size=batch_size, val_fraction=val_fraction)
    model = CharLanguageModel(
        len(vocab),
        rnn_type=rnn_type,
        embed_dim=embed_dim,
        hidden_size=hidden_size,
        num_layers=num_layers,
        fc_hidden=fc_hidden,
        dropout=dropout,
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * y.numel()
            seen += y.numel()
        train_loss = running_loss / seen
        val_loss, val_acc, val_ppl = evaluate(model, val_loader, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_perplexity": val_ppl,
            }
        )
        print(
            f"{label or rnn_type} | seq={seq_len} | epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    train_time = time.perf_counter() - start
    inference_start = time.perf_counter()
    evaluate(model, val_loader, criterion)
    inference_time = time.perf_counter() - inference_start
    final = history[-1]
    result = {
        "label": label or f"{rnn_type}_seq{seq_len}",
        "model": rnn_type.upper(),
        "seq_len": seq_len,
        "epochs": epochs,
        "embed_dim": embed_dim,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "fc_hidden": fc_hidden,
        "dropout": dropout,
        "train_loss": final["train_loss"],
        "val_loss": final["val_loss"],
        "val_accuracy": final["val_accuracy"],
        "val_perplexity": final["val_perplexity"],
        "training_time_sec": train_time,
        "inference_time_sec": inference_time,
        "parameters": count_parameters(model),
        "model_size_mb": model_size_mb(model),
        "approx_madds_per_sequence": approximate_recurrent_madds_per_sequence(
            rnn_type, seq_len, embed_dim, hidden_size, num_layers, len(vocab), fc_hidden
        ),
        "vocab_size": len(vocab),
        "history": history,
    }
    return model, vocab, result


@torch.no_grad()
def generate_text(model, vocab, prompt, length=300, temperature=0.8):
    model.eval()
    prompt = "".join(ch for ch in prompt if ch in vocab.stoi)
    if not prompt:
        prompt = vocab.chars[0]
    ids = torch.tensor([[vocab.stoi[ch] for ch in prompt]], dtype=torch.long, device=device)
    output = list(prompt)
    logits, hidden = model(ids)
    for _ in range(length):
        logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        output.append(vocab.itos[int(next_id.item())])
        logits, hidden = model(next_id, hidden)
    return "".join(output)


def plot_history(history, title):
    df = pd.DataFrame(history)
    plt.figure(figsize=(7, 4))
    plt.plot(df["epoch"], df["train_loss"], label="train loss")
    plt.plot(df["epoch"], df["val_loss"], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


**Context.** The Problem 1 section begins the assigned paragraph experiment from the PDF instructions.


## Problem 1 Experiments

**Context.** This cell stores the full assigned sequence and reports its length and vocabulary size. Those values define the input/output vocabulary for next-character prediction.


In [4]:
problem1_text = """Next character prediction is a fundamental task in the field of natural language processing (NLP) that involves predicting the next character in a sequence of text based on the characters that precede it. This task is essential for various applications, including text auto-completion, spell checking, and even in the development of sophisticated AI models capable of generating human-like text.

At its core, next character prediction relies on statistical models or deep learning algorithms to analyze a given sequence of text and predict which character is most likely to follow. These predictions are based on patterns and relationships learned from large datasets of text during the training phase of the model.

One of the most popular approaches to next character prediction involves the use of Recurrent Neural Networks (RNNs), and more specifically, a variant called Long Short-Term Memory (LSTM) networks. RNNs are particularly well-suited for sequential data like text, as they can maintain information in 'memory' about previous characters to inform the prediction of the next character. LSTM networks enhance this capability by being able to remember long-term dependencies, making them even more effective for next character prediction tasks.

Training a model for next character prediction involves feeding it large amounts of text data, allowing it to learn the probability of each character's appearance following a sequence of characters. During this training process, the model adjusts its parameters to minimize the difference between its predictions and the actual outcomes, thus improving its predictive accuracy over time.

Once trained, the model can be used to predict the next character in a given piece of text by considering the sequence of characters that precede it. This can enhance user experience in text editing software, improve efficiency in coding environments with auto-completion features, and enable more natural interactions with AI-based chatbots and virtual assistants.

In summary, next character prediction plays a crucial role in enhancing the capabilities of various NLP applications, making text-based interactions more efficient, accurate, and human-like. Through the use of advanced machine learning models like RNNs and LSTMs, next character prediction continues to evolve, opening new possibilities for the future of text-based technology."""

print(f"Problem 1 text length: {len(problem1_text):,} characters")
print(f"Problem 1 unique characters: {len(set(problem1_text))}")

Problem 1 text length: 2,391 characters
Problem 1 unique characters: 45


**Context.** This experiment cell trains `nn.RNN`, `nn.LSTM`, and `nn.GRU` at sequence lengths 10, 20, and 30. The final table is the main Problem 1 evidence for comparing training loss, validation accuracy, runtime, model size, and computational complexity.


In [5]:
P1_EPOCHS = 40
P1_BATCH_SIZE = 64
P1_EMBED_DIM = 64
P1_HIDDEN_SIZE = 128
P1_NUM_LAYERS = 1
P1_LR = 0.003
p1_results = []
p1_models = {}
for seq_len in [10, 20, 30]:
    for rnn_type in ["RNN", "LSTM", "GRU"]:
        label = f"P1_{rnn_type}_seq{seq_len}"
        model, vocab, result = train_one_experiment(
            problem1_text,
            seq_len,
            rnn_type,
            epochs=P1_EPOCHS,
            batch_size=P1_BATCH_SIZE,
            embed_dim=P1_EMBED_DIM,
            hidden_size=P1_HIDDEN_SIZE,
            num_layers=P1_NUM_LAYERS,
            lr=P1_LR,
            val_fraction=0.2,
            label=label,
        )
        p1_results.append(result)
        p1_models[label] = (model, vocab)

p1_summary = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in p1_results])
p1_summary = p1_summary.sort_values(["seq_len", "model"]).reset_index(drop=True)
p1_summary.to_csv(OUTPUT_DIR / "problem1_summary.csv", index=False)
p1_summary[
    [
        "model",
        "seq_len",
        "train_loss",
        "val_accuracy",
        "val_perplexity",
        "training_time_sec",
        "inference_time_sec",
        "parameters",
        "model_size_mb",
        "approx_madds_per_sequence",
    ]
]


P1_RNN_seq10 | seq=10 | epoch 01/40 | train_loss=2.6362 | val_loss=2.4366 | val_acc=0.3198
P1_RNN_seq10 | seq=10 | epoch 02/40 | train_loss=1.9135 | val_loss=2.2274 | val_acc=0.3937
P1_RNN_seq10 | seq=10 | epoch 03/40 | train_loss=1.6033 | val_loss=2.1692 | val_acc=0.4171
P1_RNN_seq10 | seq=10 | epoch 04/40 | train_loss=1.3694 | val_loss=2.1436 | val_acc=0.4384
P1_RNN_seq10 | seq=10 | epoch 05/40 | train_loss=1.1860 | val_loss=2.1685 | val_acc=0.4514
P1_RNN_seq10 | seq=10 | epoch 06/40 | train_loss=1.0481 | val_loss=2.1884 | val_acc=0.4622
P1_RNN_seq10 | seq=10 | epoch 07/40 | train_loss=0.9371 | val_loss=2.2398 | val_acc=0.4689
P1_RNN_seq10 | seq=10 | epoch 08/40 | train_loss=0.8508 | val_loss=2.2821 | val_acc=0.4718
P1_RNN_seq10 | seq=10 | epoch 09/40 | train_loss=0.7876 | val_loss=2.3363 | val_acc=0.4768
P1_RNN_seq10 | seq=10 | epoch 10/40 | train_loss=0.7349 | val_loss=2.4159 | val_acc=0.4706
P1_RNN_seq10 | seq=10 | epoch 11/40 | train_loss=0.6991 | val_loss=2.4535 | val_acc=0.4729

,model,seq_len,train_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence
0,GRU,10,0.483952,0.479541,19.942601,3.905722,0.009358,83181,0.317310,794880
1,LSTM,10,0.490442,0.484760,16.104954,4.327048,0.009460,108013,0.412037,1040640
2,RNN,10,0.516493,0.473486,19.578089,4.469280,0.011060,33517,0.127857,303360
3,GRU,20,0.251760,0.496973,25.772769,3.683682,0.008800,83181,0.317310,1589760
4,LSTM,20,0.258842,0.494885,21.297487,4.358862,0.009384,108013,0.412037,2081280
5,RNN,20,0.274086,0.484447,24.290012,3.575301,0.012145,33517,0.127857,606720
6,GRU,30,0.170529,0.507446,27.115401,3.900476,0.008970,83181,0.317310,2384640
7,LSTM,30,0.176747,0.505289,24.263349,4.846638,0.008970,108013,0.412037,3121920
8,RNN,30,0.188058,0.485456,31.625140,3.514459,0.007924,33517,0.127857,910080


**Context.** This analysis cell explains how to interpret the Problem 1 table. The key comparison is accuracy and perplexity versus the extra gates, parameters, and per-sequence computation of each recurrent layer type.


## Problem 1 Analysis Guide

Use the generated table to compare the requested metrics. The expected complexity pattern is:

- `nn.RNN` has the fewest gates, so it has the lowest recurrent computation and usually the fastest training time.
- `nn.GRU` uses three gates, so it is more expensive than a vanilla RNN but usually learns longer dependencies better.
- `nn.LSTM` uses four gates, so it normally has the largest parameter count and recurrent computation among the three.
- Increasing sequence length from 10 to 20 to 30 increases the work per sample approximately linearly because the recurrent cell is applied at every time step.

For the final report, cite the exact values from `p1_summary` after running the notebook. The best model should be chosen by validation accuracy and validation perplexity, not by training loss alone.


**Context.** The final table cell displays and saves the summarized metrics generated in the notebook so they can be copied into the written report.


In [6]:
all_available_tables = {}
for name in [
    "p1_summary",
    "p2_part1_summary",
    "p2_hparam_summary",
    "p2_seq50_summary",
]:
    if name in globals():
        all_available_tables[name] = globals()[name]
        print(f"\n{name}")
        display(globals()[name])

print(f"\nSaved CSV outputs in: {OUTPUT_DIR}")



p1_summary


,label,model,seq_len,epochs,embed_dim,hidden_size,num_layers,fc_hidden,dropout,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence,vocab_size
0,P1_GRU_seq10,GRU,10,40,64,128,1,0,0.0,0.483952,2.992858,0.479541,19.942601,3.905722,0.009358,83181,0.317310,794880,45
1,P1_LSTM_seq10,LSTM,10,40,64,128,1,0,0.0,0.490442,2.779127,0.484760,16.104954,4.327048,0.009460,108013,0.412037,1040640,45
2,P1_RNN_seq10,RNN,10,40,64,128,1,0,0.0,0.516493,2.974411,0.473486,19.578089,4.469280,0.011060,33517,0.127857,303360,45
3,P1_GRU_seq20,GRU,20,40,64,128,1,0,0.0,0.251760,3.249318,0.496973,25.772769,3.683682,0.008800,83181,0.317310,1589760,45
4,P1_LSTM_seq20,LSTM,20,40,64,128,1,0,0.0,0.258842,3.058589,0.494885,21.297487,4.358862,0.009384,108013,0.412037,2081280,45
5,P1_RNN_seq20,RNN,20,40,64,128,1,0,0.0,0.274086,3.190065,0.484447,24.290012,3.575301,0.012145,33517,0.127857,606720,45
6,P1_GRU_seq30,GRU,30,40,64,128,1,0,0.0,0.170529,3.300102,0.507446,27.115401,3.900476,0.008970,83181,0.317310,2384640,45
7,P1_LSTM_seq30,LSTM,30,40,64,128,1,0,0.0,0.176747,3.188967,0.505289,24.263349,4.846638,0.008970,108013,0.412037,3121920,45
8,P1_RNN_seq30,RNN,30,40,64,128,1,0,0.0,0.188058,3.453952,0.485456,31.625140,3.514459,0.007924,33517,0.127857,910080,45



Saved CSV outputs in: /content/ECGR4106_HW2_outputs
